## Day 3: Mahalanobis Baseline Scorer

Building the first anomaly detection layer: a global baseline fit on historical, known non-fraud transactions, scored using Mahalanobis distance. This notebook is scratch/exploratory work; the finalized, tested implementation is in `app/scoring.py` as `MahalanobisScorer`.

We load the historical split only (not streaming) and filter to `Class == 0`, since the baseline should represent what normal transactions look like.

In [1]:
import pandas as pd
import numpy as np
from scipy import linalg
from scipy import spatial

path = '../data/historical.csv'
data = pd.read_csv(path)

non_fraud = data.loc[data.Class == 0]
print(non_fraud.head())

       Time        V1        V2        V3        V4        V5        V6  \
0   84458.0  1.250646 -0.364827  0.882881 -0.736266 -1.201004 -0.797896   
1  163792.0  0.174174 -0.788160  0.943251 -2.723565 -0.813509 -0.069683   
2  103227.0  0.235169  1.035324 -0.381259  0.112125  1.395606 -0.050920   
3   58665.0 -0.944975  0.277247  2.526853  0.482121 -0.652469  0.074184   
4   21900.0  0.976321 -1.143034  1.534009 -0.218958 -1.883535 -0.253200   

         V7        V8        V9  ...       V21       V22       V23       V24  \
0 -0.587605 -0.018074  1.856130  ... -0.018376  0.187262 -0.040154  0.347314   
1 -0.862373  0.000397 -1.488623  ... -0.084067  0.154017  0.099854 -1.045310   
2  0.683219 -0.108672  0.622648  ... -0.285602 -0.461095  0.185673  0.100631   
3 -0.411588  0.212353 -0.098006  ...  0.031821  0.210801  0.023711  0.401571   
4 -1.242456 -0.008155  0.975215  ...  0.447036  1.208525 -0.117088  0.611053   

        V25       V26       V27       V28  Amount  Class  
0  0.5007

### Extracting the feature matrix

Pulling `V1`-`V28` into a plain numpy array. `Time` is excluded as not behaviorally meaningful; `Amount` may be added later as a standardized feature.

In [2]:
regular = non_fraud.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy()
print(regular)
print(regular.shape)

[[ 1.25064648 -0.36482739  0.88288094 ... -0.68241326  0.09224091
   0.03428805]
 [ 0.17417448 -0.78816     0.94325062 ... -0.30294309  0.01122269
   0.04914261]
 [ 0.23516943  1.03532432 -0.38125938 ...  0.10611865  0.21775552
   0.27786713]
 ...
 [-1.82513823 -0.7880854   1.25235423 ... -0.50988141 -0.02754526
   0.1334697 ]
 [ 0.73596223 -2.0748129   0.17505405 ... -0.76689597  0.06116174
   0.08398082]
 [ 1.48511829 -0.47229157 -0.45133062 ... -0.04540619 -0.04527923
  -0.00874597]]
(199020, 28)


### Mean vector

The center of the non-fraud distribution in 28D space, the first input for Mahalanobis distance.

In [3]:
mean_vector = regular.mean(axis=0)
print(mean_vector)
print(mean_vector.shape)

[ 1.18823642e-02 -5.45136464e-03  1.05419298e-02 -7.39779080e-03
  2.33411131e-03 -1.44549491e-05  1.05259773e-02 -9.79801695e-04
  5.36446469e-03  1.15737640e-02 -5.20764491e-03  1.29027169e-02
 -2.58352037e-03  1.13355986e-02  2.46717205e-04  6.81713092e-03
  1.12424814e-02  2.34762236e-03  3.62461152e-04  6.74547491e-04
 -1.12798013e-03 -1.53714738e-03 -7.99768697e-04  3.81055541e-04
 -1.20134177e-03 -5.29173645e-04 -1.68798262e-04 -9.89486342e-04]
(28,)


### Covariance matrix

This matrix represents both the variances of the features and the covariances between the features. Geometrically, the covariance matrix is like an N-dimensional ellipsoid. The variances on the diagonals represent the length of the axes of the ellipsoid, while the covariances represent the rotation of the ellipsoid relative to the standard N-dimensional axes. Captures both the spread (diagonal) and the linear relationships (off-diagonal) between features. It lets Mahalanobis distance account for correlated/differently-scaled features unlike Euclidean distance

In [4]:
covariance = np.cov(regular, rowvar=False)
print(covariance)
print(covariance.shape)

[[ 3.71551898e+00  5.70599095e-02 -1.22514754e-01  6.35716639e-02
  -5.19516522e-02 -2.42231236e-02 -1.41214304e-01  5.60595041e-03
  -4.04105500e-02 -8.43498454e-02  3.93491626e-02 -7.91956406e-02
  -7.93797970e-04 -7.20278542e-02  2.20236992e-03 -5.61069464e-02
  -1.01722515e-01 -4.12081729e-02  1.33041625e-02 -1.06137181e-02
  -4.86881772e-03  5.78346177e-03  1.11585048e-02  2.27335056e-03
   7.01591167e-03 -7.86621729e-04 -6.24260254e-03  1.06513655e-02]
 [ 5.70599095e-02  2.67913401e+00  9.76069950e-02 -4.31496456e-02
   8.11138583e-02 -6.71929051e-03  6.20844798e-02  1.98774240e-03
   3.81715167e-02  7.56106131e-02 -3.01462504e-02  6.06033737e-02
   1.53320444e-03  4.81850715e-02  6.50941871e-03  3.76691788e-02
   6.52798890e-02  2.64846556e-02 -1.19414315e-03 -2.58207627e-02
  -1.51432297e-02  4.66461298e-03  4.01763751e-03 -1.70924782e-05
   5.08430930e-03 -7.75429756e-04 -1.65958262e-03  9.45917858e-03]
 [-1.22514754e-01  9.76069950e-02  2.13713089e+00  7.14036059e-02
  -7.495

In [5]:
print(np.allclose(covariance, covariance.T))
print(np.all(np.diag(covariance) > 0))

True
True


**Sanity Check:** covariance matrix is symmetric (`np.allclose(covariance, covariance.T)` -> `True`) and has a strictly positive diagonal (all variances > 0), consistent with a valid covariance matrix.

### Variance and correlation, derived from covariance

Sanity-checking against Day 2's correlation output requires converting covariance to correlation: dividing entry `(i,j)` by `std_i * std_j`. Deriving this explicitly (rather than calling `np.corrcoef` directly) makes the covariance-to-correlation relationship concrete.

In [6]:
variance = np.diag(covariance)
print(variance)

[3.71551898 2.67913401 2.13713089 1.95233775 1.86114673 1.77122807
 1.41158092 1.35506757 1.18977539 1.10282029 1.00517325 0.89319943
 0.98850537 0.80550765 0.83604369 0.71410707 0.56384777 0.68057528
 0.65833362 0.60098805 0.51131941 0.52556698 0.37656389 0.36701547
 0.27137377 0.23238915 0.16335824 0.10272173]


In [7]:
std_deviation = np.sqrt(variance)

correlation = covariance / std_deviation
correlation = correlation / std_deviation[:, np.newaxis]

print(correlation)
print(np.allclose(correlation, correlation.T))

[[ 1.00000000e+00  1.80852334e-02 -4.34773525e-02  2.36034975e-02
  -1.97560135e-02 -9.44241664e-03 -6.16617800e-02  2.49838310e-03
  -1.92199621e-02 -4.16698728e-02  2.03612901e-02 -4.34727701e-02
  -4.14200603e-04 -4.16347443e-02  1.24958567e-03 -3.44449235e-02
  -7.02791365e-02 -2.59140643e-02  8.50657588e-03 -7.10272315e-03
  -3.53238176e-03  4.13870137e-03  9.43358847e-03  1.94676928e-03
   6.98699512e-03 -8.46541543e-04 -8.01282051e-03  1.72410771e-02]
 [ 1.80852334e-02  1.00000000e+00  4.07913579e-02 -1.88669865e-02
   3.63251598e-02 -3.08452971e-03  3.19251494e-02  1.04323562e-03
   2.13800987e-02  4.39878938e-02 -1.83702735e-02  3.91764756e-02
   9.42135429e-04  3.28004994e-02  4.34941154e-03  2.72337388e-02
   5.31130519e-02  1.96136773e-02 -8.99158296e-04 -2.03488059e-02
  -1.29382385e-02  3.93101193e-03  3.99994664e-03 -1.72371568e-05
   5.96280783e-03 -9.82737034e-04 -2.50859835e-03  1.80312175e-02]
 [-4.34773525e-02  4.07913579e-02  1.00000000e+00  3.49564328e-02
  -3.758

### Cross-check against an independent reference

Comparing this correlation matrix against `.corr()` computed directly on the historical split (unfiltered by `Class`). The difference between the 2 reveal what correlations are present only in fraudulent transactions.

In [8]:
diff = correlation - data.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].corr().to_numpy()
print(diff)
print(diff.mean())
print(np.unravel_index(np.argmax(np.abs(diff)), diff.shape))


[[ 0.00000000e+00  2.34324398e-02 -4.83908450e-02  2.11406898e-02
  -3.24625603e-02 -6.04663486e-03 -5.60515794e-02  3.03146622e-03
  -1.94218195e-02 -4.38189535e-02  2.21938461e-02 -4.23466572e-02
   2.93151364e-04 -3.86166038e-02 -1.64770232e-03 -3.65940957e-02
  -7.33021331e-02 -2.60971351e-02  6.65518193e-03  5.71957412e-03
   1.22583875e-03  6.10535114e-04  1.17245680e-03  4.38497907e-04
   1.74940819e-03 -2.08077585e-04 -4.40447963e-03 -1.19493750e-03]
 [ 2.34324398e-02  0.00000000e+00  3.84886525e-02 -1.83313925e-02
   2.48125188e-02  4.32748055e-03  4.29566779e-02  1.71925510e-03
   1.65714758e-02  3.68432285e-02 -2.00589041e-02  3.73005090e-02
   7.68178178e-05  3.64290328e-02  1.87816326e-03  2.94863072e-02
   5.66133386e-02  1.94597023e-02 -4.28788622e-03 -5.23350810e-03
  -3.56439688e-03  1.94951579e-04 -2.66724921e-03  1.79887939e-04
  -1.35989133e-03 -2.29198473e-04  2.69675123e-03 -8.18761791e-04]
 [-4.83908450e-02  3.84886525e-02 -1.11022302e-16  3.88178774e-02
  -4.872

In [9]:
print(diff[11,16])

-0.16363699325423453


**Finding:** the largest discrepancy is the `V12`/`V17` pair (diff ≈ -0.167). `V12` and `V17` show a strong correlation (≈0.84) within fraud transactions. This makes intuitive sense; it is possible that certain features(characteristics of a transaction) are correlated in fraudulent transactions i.e. vendor location and buyer location

### Conditioning check

Before deciding how to invert this covariance matrix for Mahalanobis distance, check its condition number: a measure of how close a matrix is to singular, and therefore how possible it is to invert the matrix.

In [10]:
print(np.linalg.cond(covariance))

36.55379943588063


**Finding:** condition number ≈ 36.6: Very well-conditioned. Invertability isn't actually a concern for this matrix; the Cholesky/`solve`-based approach below is motivated more by computational efficiency.

### Cholesky factorization

Rather than computing `covariance`'s inverse directly (`np.linalg.inv`), factorize it as `covariance = L @ L.T` (where L is lower-triangular). Mahalanobis distance only ever needs $\Sigma^{-1}v$ for a deviation vector $v$, not the standalone inverse matrix — so factor once, then solve two triangular systems (forward then back substitution) per transaction. This is cheaper than a dense matrix-vector product (roughly half the operations, since triangular solves only touch the nonzero triangle). It also fails loudly on non-invertible matrices, making it safer than `inv`

In [11]:
#cov = lower @ upper = LL.T
lower = np.linalg.cholesky(covariance)
upper = lower.T

### Scoring function

Mahalanobis distance is $D(x) = \sqrt{(x-\mu)^T \Sigma^{-1} (x-\mu)}$, computed as: solve $Lz = (x-\mu)$ for $z$, then solve $L^Ty = z$ for $y$, then $D(x) = \sqrt{(x-\mu)^T y}$. This never forms $\Sigma^{-1}$ explicitly.

Note that this is similar to Euclidean distance, but with $\Sigma^{-1}$ included. For some intuition, multiply by $\Sigma^{-1}$ is similar to standardizing the calculation(such as dividing by $\sigma$ when calculating z-score), so that the outputted score represents how far from the mean $x$ is from $\mu$, standardized by how likely $x$ is to deviate from $\mu$. 

In [12]:
def scoring(x: np.ndarray) -> float:
    z = linalg.solve_triangular(lower, x - mean_vector, lower=True)
    v = linalg.solve_triangular(upper, z, lower=False)
    return np.sqrt((x - mean_vector).T @ v)

### Validating against independent references

Checking this Cholesky-based implementation against two independent computations of the same formula: a direct `np.linalg.inv`-based version, and `scipy.spatial.distance.mahalanobis`.

In [13]:
personal = [scoring(x) for x in regular[0:10]]
test1 = [np.sqrt((x-mean_vector).T @ np.linalg.inv(covariance) @ (x-mean_vector)) for x in regular[0:10]]
test2 = [spatial.distance.mahalanobis(x, mean_vector, np.linalg.inv(covariance)) for x in regular[0:10]]

print(np.array(personal))
print(np.array(test1))
print(np.array(test2))

[3.64150765 4.77929822 5.58021036 4.61477936 5.67746923 5.01298116
 5.05896872 4.41196725 2.66297099 3.02502452]
[3.64150765 4.77929822 5.58021036 4.61477936 5.67746923 5.01298116
 5.05896872 4.41196725 2.66297099 3.02502452]
[3.64150765 4.77929822 5.58021036 4.61477936 5.67746923 5.01298116
 5.05896872 4.41196725 2.66297099 3.02502452]


**Finding:** all three implementations agree across the sample rows checked. The scoring function is validated against two independent references.

### Fraud vs. non-fraud separation

A directional sanity check before building formal tests: known-fraud transactions should score meaningfully higher than known-normal transactions, if this baseline is picking up any real signal at all.

In [14]:
fraud = data.loc[data.Class == 1]
fraud = fraud.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy()

reg_scores = np.array([scoring(x) for x in regular])
fraud_scores = np.array([scoring(x) for x in fraud])

reg_mean, reg_min, reg_max = np.mean(reg_scores), np.min(reg_scores), np.max(reg_scores)
fraud_mean, fraud_min, fraud_max = np.mean(fraud_scores), np.min(fraud_scores), np.max(fraud_scores)

print(f"Regular mean: {reg_mean}, min: {reg_min}, max: {reg_max}")
print(f"Fraud mean: {fraud_mean}, min: {fraud_min}, max: {fraud_max}")

Regular mean: 4.599701765462432, min: 2.076968858712324, max: 187.62944121643298
Fraud mean: 45.826411462528604, min: 3.105662421449797, max: 128.06767799175606


**Finding:** mean score for normal transactions ≈ 4.6 vs. ≈ 45.8 for fraud transactions — roughly a 10x separation in the mean, with fraud scores skewing much higher. There's overlap at the extremes (normal max ≈ 187.6, fraud min ≈ 3.1), which is expected from a single global baseline alone, and is part of the motivation for the per-entity EWMA layer in Day 4. Directionally, though, the global Mahalanobis baseline is working as intended.

### Fraud vs. non-fraud separation (streaming set)

The previous check reused the historical split, which still contains its own fraud rows — useful as a first look, but not the actual held-out evaluation set. Re-running the same check on `streaming.csv` (truly unseen by the baseline fit) confirms the separation isn't an artifact of checking against data the baseline has any relationship to, and doubles as a check that the historical/streaming split is behaving consistently. Per the earlier discussion on leakage: this is a single, non-iterative look — the last one before `streaming.csv` is reserved untouched for Day 5's formal evaluation.

In [15]:
stream_path = '../data/streaming.csv'
data = pd.read_csv(stream_path)

regular = data.loc[data.Class == 0]
regular = regular.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy()
fraud = data.loc[data.Class == 1]
fraud = fraud.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy()

reg_scores = np.array([scoring(x) for x in regular])
fraud_scores = np.array([scoring(x) for x in fraud])

reg_mean, reg_min, reg_max = np.mean(reg_scores), np.min(reg_scores), np.max(reg_scores)
fraud_mean, fraud_min, fraud_max = np.mean(fraud_scores), np.min(fraud_scores), np.max(fraud_scores)

print(f"Regular mean: {reg_mean}, min: {reg_min}, max: {reg_max}")
print(f"Fraud mean: {fraud_mean}, min: {fraud_min}, max: {fraud_max}")

Regular mean: 4.603268288675284, min: 2.0605063142622644, max: 131.35920082097647
Fraud mean: 47.437526072012, min: 3.271699616186535, max: 113.39573379148513


**Finding:** streaming-set scores closely track the historical-set numbers — normal mean ≈ 4.6 (vs. ≈ 4.6 historical), fraud mean ≈ 47.4 (vs. ≈ 45.8 historical). The historical/streaming split is consistent, and the fraud/non-fraud separation holds up on unseen data, not just the historical split. 